# 2022 SE QLD Floods — Probability Ratio (multiplicative GEV shift-fit)

Precipitation analogue of notebook 04. The counterfactual rescales each wet-season
7-day-max precipitation total to a pre-industrial climate **multiplicatively** (log-space
shift), with a GEV fitted to the block maxima.

- **Metric**: wet-season (Nov–Apr) maximum 7-day rolling precip, area-weighted over SE QLD.
- **Shift coefficient** β = d(log precip)/d(GMST) = ln(1+CC_rate)·α_QLD.
- **Primary**: CC 7%/°C × α_QLD = 0.289 (ERA5-observed wet-season Tmax) — a conservative
  lower bound. Sensitivities: CMIP6 α_QLD = 0.882, and dynamic 14%/°C.

A data-driven fitted β is also reported but is **not** used: the SE QLD wet-season precip
record is ENSO-dominated, and the fitted slope (~0.28, i.e. 28%/°C) reflects internal
variability, not a forced thermodynamic response.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('../../').resolve()
sys.path.insert(0, str(ROOT))
from src.attribution import (
    area_weighted_series, season_block_max, wet_season_max_ndays,
    load_gmst, extrapolate_to, smoothed_covariate, event_gmst_sigma,
    shift_fit_gev, fit_gev, build_liability_table, far,
    AUD_TO_USD, CC_RATE_STANDARD, CC_RATE_HIGH, CLIM_START, CLIM_END,
)
from scipy.stats import genextreme

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
RAW  = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
FIGS = ROOT / 'outputs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)
print('Setup complete.')


In [ ]:
# ── Region / season ──
LAT_S, LAT_N = -30, -24
LON_W, LON_E = 150, 154
WET_MONTHS = [11, 12, 1, 2, 3, 4]
EVENT_YEAR = 2022
ERA5_TP_PATH = RAW / 'era5' / 'era5_tp_daily_se_qld_1961_2022.nc'

ds_tp = xr.open_dataset(ERA5_TP_PATH)
if 'valid_time' in ds_tp.coords and 'time' not in ds_tp.dims:
    ds_tp = ds_tp.rename({'valid_time': 'time'})
tp_var = next(v for v in ds_tp.data_vars if 'tp' in v.lower() or 'precip' in v.lower())
da_tp = ds_tp[tp_var]

# ERA5 tp: 1-hour accumulation (m), sampled 4×/day. Sum to daily and scale to mm/day.
# The scale factor is common to all seasons, so it cancels in the multiplicative PR.
n_expected = 62 * len(WET_MONTHS) * 31
n_per_day  = da_tp.time.size / n_expected
da_daily   = da_tp.resample(time='1D').sum() * (24 / n_per_day) * 1000
ts_tp = area_weighted_series(da_daily)

ws = wet_season_max_ndays(ts_tp, range(CLIM_START + 1, EVENT_YEAR + 1), ndays=7)
rank = sorted(ws.values, reverse=True).index(ws[EVENT_YEAR]) + 1
print(f'Wet-season 7-day-max precip: {len(ws)} seasons')
print(f'{EVENT_YEAR} = {ws[EVENT_YEAR]:.1f} mm  (rank {rank}/{len(ws)})')


In [ ]:
# ── GMST covariate (extrapolated to 2022, FaIR ends 2021) ──
gmst = load_gmst(PROC)
covariate = smoothed_covariate(extrapolate_to(gmst['t_p50'], EVENT_YEAR))
g_sigma   = event_gmst_sigma(gmst, EVENT_YEAR)
print(f'GMST({EVENT_YEAR}) = {covariate.loc[EVENT_YEAR]:.3f} °C vs pre-industrial (±{g_sigma:.3f})')

# ── α_QLD and shift coefficients β_log = ln(1+CC)·α ──
qld_af  = pd.read_csv(PROC / 'qld_amplification_factor.csv').set_index('model')
if 'ERA5_observed' not in qld_af.index:
    raise KeyError(
        "qld_amplification_factor.csv is missing the 'ERA5_observed' row (α=0.289) — the PRIMARY "
        "α_QLD for this notebook. It is not recomputed here; notebook 06's save cell preserves it. "
        "Rerun notebook 06 (which now preserves the row) or restore it. "
        "See wiki/findings/2026-06-17-lei-dropna-fix.md."
    )
A_ERA5  = float(qld_af.loc['ERA5_observed', 'amplification'])     # 0.289
A_CMIP6 = float(qld_af.drop(index='ERA5_observed')['amplification'].median())  # 0.882
beta = lambda cc, a: float(np.log(1 + cc) * a)
print(f'α_QLD ERA5-observed = {A_ERA5:.3f};  α_QLD CMIP6 median = {A_CMIP6:.3f}')


In [ ]:
# ── Multiplicative shift-fit PR ──
methods = {
    'primary (CC 7%/°C × α=0.289)':  beta(CC_RATE_STANDARD, A_ERA5),
    'sens (CC 7%/°C × α=0.882 CMIP6)': beta(CC_RATE_STANDARD, A_CMIP6),
    'sens (CC 14%/°C × α=0.289)':    beta(CC_RATE_HIGH, A_ERA5),
    'sens (β fitted, ENSO-contaminated)': None,
}
results = {}
for name, b in methods.items():
    r = shift_fit_gev(ws, covariate, EVENT_YEAR, mode='multiplicative',
                      beta=b, g_event_sigma=g_sigma)
    results[name] = r
    tag = f'(fitted β={r.beta:.3f})' if b is None else f'(β={r.beta:.4f})'
    print(f'{name:36s}: PR={r.pr:.2f} [{r.pr_p05:.2f}–{r.pr_p95:.2f}] FAR={r.far:.3f} {tag}')

primary = results['primary (CC 7%/°C × α=0.289)']
print(f'\nPRIMARY  PR = {primary.pr:.2f} [{primary.pr_p05:.2f}–{primary.pr_p95:.2f}]  '
      f'FAR = {primary.far:.3f}  (conservative lower bound)')


In [ ]:
# ── Persist PR table + bootstrap ──
rows = [{'method': n, 'beta': r.beta, 'pr': r.pr, 'pr_p05': r.pr_p05,
         'pr_p95': r.pr_p95, 'far': r.far, 'gev_xi': -r.gev_params[0]}
        for n, r in results.items()]
pr_df = pd.DataFrame(rows)
pr_df.to_csv(PROC / 'qld_floods_pr_era5.csv', index=False)
pd.DataFrame({'pr_boot': primary.pr_boot}).to_parquet(
    PROC / 'qld_floods_pr_shiftfit_bootstrap.parquet', index=False)
print('Saved qld_floods_pr_era5.csv and qld_floods_pr_shiftfit_bootstrap.parquet')
print(pr_df.to_string(index=False))


In [ ]:
# ── Figure ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
xi, loc, scale = primary.gev_params
factor = np.exp(primary.beta * covariate.loc[EVENT_YEAR])
x = np.linspace(ws.min() * 0.6, ws.max() * 1.1, 300)

ax = axes[0]
ax.hist(ws.values, bins=16, density=True, alpha=0.35, color='#90A4AE', label='ERA5 seasons')
ax.plot(x, genextreme.pdf(x, xi, loc, scale), color='#2196F3', lw=2.5,
        label='P1 factual GEV (2022)')
ax.plot(x, genextreme.pdf(x, xi, loc / factor, scale / factor), color='#4CAF50', lw=2.5,
        label='P0 counterfactual GEV')
ax.axvline(primary.threshold, color='k', ls='--', lw=1.5,
           label=f'2022 event ({primary.threshold:.0f} mm)')
ax.set_xlabel('Wet-season 7-day-max precip (mm)')
ax.set_ylabel('Density')
ax.set_title('Multiplicative shift-fit GEV', fontsize=11)
ax.legend(fontsize=8)

ax2 = axes[1]
boot = primary.pr_boot; boot = boot[np.isfinite(boot)]
ax2.hist(np.clip(boot, 0, 5), bins=40, density=True, alpha=0.7, color='#673AB7')
ax2.axvline(primary.pr, color='k', lw=2, ls='--', label=f'median PR = {primary.pr:.2f}')
ax2.axvline(1.0, color='grey', lw=1, ls=':')
ax2.set_xlabel('Probability Ratio')
ax2.set_title('Bootstrap PR distribution', fontsize=11)
ax2.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGS / 'qld_floods_pr_shiftfit.png', bbox_inches='tight')
plt.show()
print('Saved figure.')
